# Importing Libs

In [1]:
!pip install langchain_community
!pip install langchain_huggingface
!pip install langchain_groq
!pip install faiss-cpu

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

# Preprocessing

In [3]:
import json

with open("hotpot_subset.json") as f:
    data = json.load(f)

contexts = [item["context"] for item in data]
questions = [item["question"] for item in data]
answers = [item["answer"]for item in data]

In [4]:
def extract_context(example):
    titles = example["title"]
    sentences = example["sentences"]

    docs = []
    for title, sent_list in zip(titles, sentences):
        text = title + " " + " ".join(sent_list)
        docs.append(text)

    return docs

In [5]:
all_contexts = []

for item in contexts:
    docs = extract_context(item)
    all_contexts.extend(docs)

# Chunking

In [7]:
from langchain_classic.schema import Document

docs = [Document(page_content=text) for text in all_contexts]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunked_contexts = splitter.split_documents(docs)

# Embedding

In [6]:
embeddings = HuggingFaceEmbeddings( model_name = "all-MiniLM-L6-V2" )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-V2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# VectorStore and Retrieval

In [9]:
import os

FAISS_PATH = "/content/"

if os.path.exists(FAISS_PATH):
    vectorstore = FAISS.load_local(
        FAISS_PATH,
        embeddings,
        allow_dangerous_deserialization=True
    )
    print("Loaded in seconds!")
else:
    print("Building index, ~12 mins, only once...")
    vectorstore = FAISS.from_documents(chunked_contexts, embeddings)
    vectorstore.save_local(FAISS_PATH)
    print("Built and saved!")

Loaded in seconds!


In [10]:
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

# LLM

In [11]:
from google.colab import userdata
api_key = userdata.get("groq_api_key_2")


llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=api_key
)

# Rag Func

In [12]:
def simple_rag(question):

    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are a question answering system.

Use ONLY the context to answer.

Rules:
- Return ONLY the short answer span.
- Do NOT explain.
- Do NOT write full sentences.
- Answer must be a phrase from the context.

Context:
{context}

Question:
{question}

Answer:
"""

    response = llm.invoke(prompt)

    return response.content

In [13]:
print(simple_rag(questions[0]))

American


In [14]:
answers[0]

'yes'

In [15]:
print(simple_rag(questions[197]))

# Evaluation

In [16]:
predictions = []

In [41]:
import time



for q in questions[75:100]:

    while True:
        try:
            pred = simple_rag(q)
            predictions.append(pred)
            time.sleep(6)
            break
        except Exception:
            print("Rate limit hit, waiting...")
            time.sleep(10)

In [42]:
predictions

['American',
 '',
 'Deviants',
 'No',
 '',
 'YG Entertainment',
 'Eenasul Fateh',
 '1,400 people',
 'Annie Morton',
 'American rock band',
 'Kansas Song (We’re From Kansas)',
 'David Weissman',
 '1996',
 'mixed-use tower',
 'from 1986 to 2013',
 '',
 'America East Conference',
 'opera composers',
 'Nixon',
 'Robert Erskine Childers',
 'Pedro Rodríguez',
 'Miles "Tails" Prower',
 'nameless IR remotes',
 '',
 '',
 'Lee Hazlewood',
 '',
 'a genus',
 'Henry J. Kaiser',
 'Crusaders of Khazan',
 'March 14, 2000',
 'Fujioka, Gunma',
 'Charles Nungesser',
 '',
 'Screaming Trees',
 'the Russian Civil War',
 '2000',
 'World War\u202fII',
 'Nevada',
 'Columbia University',
 'Scotch Collie',
 '',
 '1980',
 'ensuring its sovereignty and freedom from colonization',
 'Vice President Biden',
 'North Berwick',
 'Buddy Holly',
 '',
 'genus of flowering plants',
 'English Electric Canberra',
 '2014–15 Pac-12 Conference',
 '1,462 hypermarkets',
 'Indianapolis Motor Speedway',
 '',
 '',
 'Adelaide',
 'drif

In [43]:
len(predictions)

100

In [29]:
print(simple_rag(questions[197]))

In [21]:
import re
import string

def normalize_text(text):
    text = text.lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = ''.join(ch for ch in text if ch not in string.punctuation)
    text = ' '.join(text.split())
    return text

## Exact Match

In [22]:
def exact_match_score(prediction, ground_truth):
    return normalize_text(prediction) == normalize_text(ground_truth)

## F1 Score

In [23]:
def f1_score(prediction, ground_truth):

    pred_tokens = normalize_text(prediction).split()
    truth_tokens = normalize_text(ground_truth).split()

    common = set(pred_tokens) & set(truth_tokens)

    if len(common) == 0:
        return 0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(truth_tokens)

    return 2 * (precision * recall) / (precision + recall)

## Eval func

In [24]:
def evaluate(predictions, subset_ans):

    total = len(predictions)
    exact_match = 0
    f1 = 0

    for pred, truth in zip(predictions, subset_ans):

        if exact_match_score(pred, truth):
            exact_match += 1

        f1 += f1_score(pred, truth)

    exact_match = exact_match / total
    f1 = f1 / total

    return {
        "Exact Match": exact_match,
        "F1 Score": f1
    }

In [45]:
subset_ans=answers[:100]
evaluate(predictions, subset_ans)

{'Exact Match': 0.3, 'F1 Score': 0.4106428571428571}